# 02 — Enzyme–Protein Screening

This notebook expands the initial HERALD digestion workflow from a single protein–enzyme example to a small screening workflow.

The goal is to evaluate all currently defined whey proteins against all currently defined enzymes and summarize which protein–enzyme combinations produce the most AMP-like peptide profiles.

This notebook uses the current rule-based `simple_amp_score` as an exploratory baseline. The score is not a validated AMP predictor, but it provides a transparent first-pass ranking method for comparing digestion outputs.

In [34]:
import sys
import os
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from herald.proteins import protein_sequence, WHEY_PROTEINS
from herald.enzymes import ENZYME_RULES
from herald.digestion import digest_sequence
from herald.scoring import peptide_features

## Define Screening Scope

This section defines the proteins and enzymes included in the initial screen.

The protein set comes from `WHEY_PROTEINS`, which stores target whey proteins and their UniProt accession IDs. The enzyme set comes from `ENZYME_RULES`, which currently contains the proteases with defined cleavage rules.

At this stage, the screen is intentionally small and used to test the workflow. Additional food-grade or commercially available enzymes can be added later.

In [5]:
print("Proteins:")
for protein_name, accession_id in WHEY_PROTEINS.items():
    print(f"- {protein_name}: {accession_id}")

print("\nEnzymes:")
for enzyme_name in ENZYME_RULES.keys():
    print(f"- {enzyme_name}")

Proteins:
- beta-lactoglobulin: P02754
- alpha-lactalbumin: P00709
- lactoferrin: P24627
- BSA: P02769

Enzymes:
- trypsin
- chymotrypsin


## Run Enzyme–Protein Screening

This section runs the current HERALD workflow across every protein–enzyme combination.

For each combination, the workflow:

1. retrieves the protein sequence,
2. simulates enzymatic digestion,
3. computes AMP-like peptide features,
4. ranks the resulting peptides, and
5. stores summary statistics for comparison.

The output is a list of screening results that can later be converted into a table.

In [45]:
results = []

for protein_name, accession_id in WHEY_PROTEINS.items():
    sequence = protein_sequence(accession_id)

    for enzyme_name in ENZYME_RULES.keys():
        
        peptides = digest_sequence(sequence, enzyme_name)
        
        features = []
        for peptide in peptides:
            features.append(peptide_features(peptide))
            
        scores = []
        for feature in features:
            scores.append(feature["simple_amp_score"])

        if len(scores) == 0:
            continue

        top_score = max(scores)
        top_index = scores.index(top_score)
        top_peptide = peptides[top_index]

        num_score_ge_3 = 0
        num_score_ge_4 = 0
        for score in scores:
            if score >= 3:
                num_score_ge_3 += 1
            if score >= 4:
                num_score_ge_4 += 1

        avg_score = sum(scores)/len(scores)
            
        summary = {
            "protein": protein_name,
            "enzyme": enzyme_name,
            "num_peptides": len(peptides),
            "max_score": top_score,
            "top_peptide": top_peptide,
            "num_score_ge_3": num_score_ge_3,
            "num_score_ge_4": num_score_ge_4,
            "avg_score": avg_score,
        }

        results.append(summary)
results

[{'protein': 'beta-lactoglobulin',
  'enzyme': 'trypsin',
  'num_peptides': 13,
  'max_score': 3,
  'top_peptide': 'CLLLALALTCGAQALIVTQTMK',
  'num_score_ge_3': 7,
  'num_score_ge_4': 0,
  'avg_score': 2.3076923076923075},
 {'protein': 'beta-lactoglobulin',
  'enzyme': 'chymotrypsin',
  'num_peptides': 19,
  'max_score': 4,
  'top_peptide': 'ENGECAQKKIIAEKTKIPAVF',
  'num_score_ge_3': 4,
  'num_score_ge_4': 2,
  'avg_score': 1.5789473684210527},
 {'protein': 'alpha-lactalbumin',
  'enzyme': 'trypsin',
  'num_peptides': 12,
  'max_score': 4,
  'top_peptide': 'GIDYWLAHK',
  'num_score_ge_3': 5,
  'num_score_ge_4': 1,
  'avg_score': 2.25},
 {'protein': 'alpha-lactalbumin',
  'enzyme': 'chymotrypsin',
  'num_peptides': 16,
  'max_score': 3,
  'top_peptide': 'AKQF',
  'num_score_ge_3': 3,
  'num_score_ge_4': 0,
  'avg_score': 1.625},
 {'protein': 'lactoferrin',
  'enzyme': 'trypsin',
  'num_peptides': 58,
  'max_score': 5,
  'top_peptide': 'QVLLHQQALFGK',
  'num_score_ge_3': 34,
  'num_scor

## Convert Screening Results to a Table

The screening loop stores one summary dictionary for each protein–enzyme combination. To compare combinations more easily, we convert the list of dictionaries into a pandas DataFrame.

This table lets us sort combinations by the number of high-scoring peptides, maximum score, average score, and top peptide sequence.

In [48]:
results_df = pd.DataFrame(results)
results_df

,protein,enzyme,num_peptides,max_score,top_peptide,num_score_ge_3,num_score_ge_4,avg_score
0,beta-lactoglobulin,trypsin,13,3,CLLLALALTCGAQALIVTQTMK,7,0,2.307692
1,beta-lactoglobulin,chymotrypsin,19,4,ENGECAQKKIIAEKTKIPAVF,4,2,1.578947
2,alpha-lactalbumin,trypsin,12,4,GIDYWLAHK,5,1,2.250000
3,alpha-lactalbumin,chymotrypsin,16,3,AKQF,3,0,1.625000
4,lactoferrin,trypsin,58,5,QVLLHQQALFGK,34,9,2.465517
5,lactoferrin,chymotrypsin,67,5,AAPRKNVRW,26,7,2.134328
6,BSA,trypsin,62,4,WVTFISLLLLFSSAYSR,21,3,1.951613
7,BSA,chymotrypsin,61,5,EIARRHPY,17,5,1.901639


In [49]:
ranked_results_df = results_df.sort_values(
    by=["num_score_ge_4", "num_score_ge_3", "max_score", "avg_score"],
    ascending=False,
)

ranked_results_df

,protein,enzyme,num_peptides,max_score,top_peptide,num_score_ge_3,num_score_ge_4,avg_score
4,lactoferrin,trypsin,58,5,QVLLHQQALFGK,34,9,2.465517
5,lactoferrin,chymotrypsin,67,5,AAPRKNVRW,26,7,2.134328
7,BSA,chymotrypsin,61,5,EIARRHPY,17,5,1.901639
6,BSA,trypsin,62,4,WVTFISLLLLFSSAYSR,21,3,1.951613
1,beta-lactoglobulin,chymotrypsin,19,4,ENGECAQKKIIAEKTKIPAVF,4,2,1.578947
2,alpha-lactalbumin,trypsin,12,4,GIDYWLAHK,5,1,2.250000
0,beta-lactoglobulin,trypsin,13,3,CLLLALALTCGAQALIVTQTMK,7,0,2.307692
3,alpha-lactalbumin,chymotrypsin,16,3,AKQF,3,0,1.625000


## Save Ranked Screening Results

The ranked enzyme–protein screening table is saved as a CSV file in `data/processed/`.

This creates a reusable output that can be inspected later, shared with collaborators, or used as input for downstream analysis.

In [51]:
PROJECT_ROOT = os.path.abspath("..")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

output_path = os.path.join(
    PROCESSED_DATA_DIR,
    "enzyme_protein_screening_results.csv",
)

ranked_results_df.to_csv(output_path, index=False)

print(f"Saved results to: {output_path}")

Saved results to: /Users/lukas/Developer/herald/data/processed/enzyme_protein_screening_results.csv
